In [ ]:
import pandas as pd
import os
import re
import shutil
from datetime import datetime
import win32com.client as win32

# ==========================
# CONFIGURAÇÕES
# ==========================

CAMINHO_BASE     = r'C:\Automacao_PE\Extracao_PE_Produto_Foco_TESTE.xlsx'
CAMINHO_MODELO   = r'C:\Automacao_PE\Modelo_Envio_PE.xlsx'   # Modelo com abas BASE_PE + TT_PE
PASTA_SAIDA      = r'C:\Automacao_PE\Arquivos_Gerados'
ARQUIVO_LOG      = r'C:\Automacao_PE\LOG_ENVIO_PE.xlsx'

# Aba com os dados na extração
ABA_DADOS        = 'BASE_PE'

# Abas do modelo
ABA_MODELO_BASE  = 'BASE_PE'   # aba onde os dados filtrados serão injetados
ABA_MODELO_PIVOT = 'TT_PE'    # aba com a tabela dinâmica (atualizada automaticamente)

# Colunas que existem no modelo (sem as colunas de roteamento especialista/gerente)
COLUNAS_MODELO = [
    'COD_PESQUISA', 'DATA', 'COD_LOJA', 'COD_SAP', 'NOME_FANTASIA',
    'BANDEIRA', 'DES_REGIAO', 'NOM_PESSOA_COMPLETO', 'DES_CATEGORIA',
    'DES_SUB_CATEGORIA', 'DES_TIPO_PONTO_EXTRA', 'FOTO', 'CHAVE',
    'EXISTE', 'PE_RETORNO'
]

EMAILS_CC = [
    'coordenador@empresa.com',
    'diretoria@empresa.com'
]

# Número de tentativas de reenvio em caso de falha no Outlook
MAX_TENTATIVAS = 2

# ==========================
# MODO TESTE
# ==========================
# True  → gera arquivos de Especialistas E Gerentes sem enviar nenhum e-mail.
# False → executa o processo completo (gera + envia e-mails).

MODO_TESTE = True


# ==========================
# HELPER
# ==========================

def tipo_label(modo_teste):
    return "TESTE" if modo_teste else "PRODUÇÃO"


# ==========================
# PREPARAÇÃO
# ==========================

df = pd.read_excel(CAMINHO_BASE, sheet_name=ABA_DADOS)

os.makedirs(PASTA_SAIDA, exist_ok=True)

data_log  = datetime.now().strftime('%d-%m-%Y %H:%M:%S')
data_nome = datetime.now().strftime('%d-%m-%Y')


def limpar_nome(nome):
    """Remove caracteres inválidos para nome de arquivo."""
    return re.sub(r'[\\/*?:"<>|]', "", str(nome))


# ==========================
# VALIDAÇÃO DO MODELO EXCEL
# ==========================

def validar_modelo():
    """Verifica se o modelo existe e possui as abas necessárias antes de iniciar o processo."""
    if not os.path.exists(CAMINHO_MODELO):
        raise FileNotFoundError(f"Modelo não encontrado: {CAMINHO_MODELO}")

    import openpyxl
    wb = openpyxl.load_workbook(CAMINHO_MODELO, read_only=True)
    abas_ausentes = [aba for aba in [ABA_MODELO_BASE, ABA_MODELO_PIVOT] if aba not in wb.sheetnames]
    wb.close()

    if abas_ausentes:
        raise ValueError(f"Abas ausentes no modelo: {abas_ausentes}")

    print("✅ Modelo validado com sucesso.")


# ==========================
# LOG
# ==========================

# Acumula todos os registros em memória e salva uma única vez ao final,
# evitando reescrever o arquivo a cada entrada (mais rápido e seguro).
registros_log = []

def registrar_log(tipo, nome, email, status, erro=""):
    """Acumula log em memória."""
    registros_log.append({
        "DATA_ENVIO": data_log,
        "TIPO":       tipo,
        "NOME":       nome,
        "EMAIL":      email,
        "STATUS":     status,
        "ERRO":       erro
    })


def salvar_log():
    """Persiste todos os registros acumulados no arquivo de log Excel."""
    if not registros_log:
        return

    novo = pd.DataFrame(registros_log)

    if os.path.exists(ARQUIVO_LOG):
        antigo = pd.read_excel(ARQUIVO_LOG)
        final  = pd.concat([antigo, novo], ignore_index=True)
    else:
        final = novo

    final.to_excel(ARQUIVO_LOG, index=False)
    print(f"\n📋 Log salvo: {len(registros_log)} registros em {ARQUIVO_LOG}")


# ==========================
# GERAR RELATÓRIO COM MODELO
# ==========================

def gerar_relatorio(df_filtrado, nome_saida, excel_app):
    """
    Recebe a instância do Excel já iniciada (reutilizada entre chamadas).
    1. Copia o modelo para o destino final (preserva o original intacto).
    2. Abre a cópia e injeta os dados em bloco via COM (escrita vetorizada).
    3. Atualiza todas as tabelas dinâmicas via RefreshAll.
    4. Salva e fecha apenas o workbook (não encerra o Excel).
    """
    caminho_final = os.path.abspath(nome_saida)
    shutil.copy2(CAMINHO_MODELO, caminho_final)

    wb = None
    try:
        wb      = excel_app.Workbooks.Open(caminho_final)
        ws_base = wb.Sheets(ABA_MODELO_BASE)

        # --- Limpa dados anteriores mantendo o cabeçalho (linha 1) ---
        ultima_linha = ws_base.Cells(ws_base.Rows.Count, 1).End(-4162).Row  # xlUp
        if ultima_linha > 1:
            ws_base.Range(
                ws_base.Cells(2, 1),
                ws_base.Cells(ultima_linha, len(COLUNAS_MODELO))
            ).ClearContents()

        # --- Garante o cabeçalho correto na linha 1 ---
        for col_idx, col_nome in enumerate(COLUNAS_MODELO, start=1):
            ws_base.Cells(1, col_idx).Value = col_nome

        # --- Seleciona apenas as colunas que existem no modelo ---
        colunas_presentes = [c for c in COLUNAS_MODELO if c in df_filtrado.columns]
        df_para_gravar    = df_filtrado[colunas_presentes].copy()

        # --- Escrita em bloco (vetorizada): substitui o loop linha a linha ---
        # Converte NaN para None (COM não aceita float NaN)
        dados = [
            [None if pd.isna(v) else v for v in row]
            for row in df_para_gravar.itertuples(index=False)
        ]
        if dados:
            ws_base.Range(
                ws_base.Cells(2, 1),
                ws_base.Cells(1 + len(dados), len(colunas_presentes))
            ).Value = dados

        # --- Atualiza a tabela dinâmica TT_PE e quaisquer outras ---
        wb.RefreshAll()
        excel_app.CalculateUntilAsyncQueriesDone()  # aguarda o refresh terminar antes de salvar

        wb.Save()

    finally:
        if wb is not None:
            try:
                wb.Close(SaveChanges=False)
            except Exception:
                pass


# ==========================
# ENVIAR E-MAIL OUTLOOK
# ==========================

def enviar_email(destinatario, nome, arquivo, tipo, outlook_app):
    """Envia e-mail via Outlook com retry. Usa a instância recebida por parâmetro."""
    if pd.isna(destinatario) or str(destinatario).strip() == "":
        registrar_log(tipo, nome, destinatario, "ERRO", "Sem e-mail")
        return

    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            mail = outlook_app.CreateItem(0)

            mail.To      = str(destinatario).strip()
            mail.CC      = "; ".join(EMAILS_CC)
            mail.Subject = f"Relatório Semanal - Pontos Extras ({data_nome})"
            mail.Body    = (
                f"Olá {nome},\n\n"
                f"Segue relatório atualizado de Pontos Extras ({tipo}).\n\n"
                f"Qualquer dúvida fico à disposição.\n\n"
                f"Atenciosamente,"
            )

            mail.Attachments.Add(os.path.abspath(arquivo))
            mail.Send()

            registrar_log(tipo, nome, destinatario, "ENVIADO")
            return

        except Exception as e:
            if tentativa < MAX_TENTATIVAS:
                print(f"    ⚠️ Tentativa {tentativa} falhou para {nome}, tentando novamente...")
            else:
                registrar_log(tipo, nome, destinatario, "ERRO", str(e))
                print(f"    ❌ ERRO após {MAX_TENTATIVAS} tentativas: {e}")


# ==========================
# VALIDAÇÃO DAS COLUNAS
# ==========================

colunas_necessarias = [
    'ESPECIALISTA', 'EMAIL ESPECIALISTA',
    'GERENTE',      'EMAIL GERENTE'
]
ausentes = [c for c in colunas_necessarias if c not in df.columns]
if ausentes:
    raise ValueError(f"Colunas ausentes no arquivo base: {ausentes}")

# Valida o modelo antes de iniciar qualquer processamento
validar_modelo()

print(f"Base carregada: {len(df)} registros | "
      f"{df['ESPECIALISTA'].nunique()} especialistas | "
      f"{df['GERENTE'].nunique()} gerentes")


# ==========================
# CONFIRMAÇÃO EM PRODUÇÃO
# ==========================

if not MODO_TESTE:
    total_dest = df['ESPECIALISTA'].nunique() + df['GERENTE'].nunique()
    resposta = input(
        f"\n⚠️  MODO PRODUÇÃO — serão enviados e-mails para {total_dest} destinatários."
        f"\nConfirma o envio? [s/n]: "
    ).strip().lower()
    if resposta != 's':
        print("Envio cancelado pelo usuário.")
        raise SystemExit(0)

if MODO_TESTE:
    print("\n⚠️  MODO TESTE ATIVADO — arquivos de ESPECIALISTA e GERENTE serão gerados. Nenhum e-mail será enviado.\n")
    PASTA_ATUAL = os.path.join(PASTA_SAIDA, 'TESTE')
    os.makedirs(PASTA_ATUAL, exist_ok=True)
else:
    PASTA_ATUAL = PASTA_SAIDA


# ==========================
# INICIALIZA INSTÂNCIAS COM
# ==========================

# Uma única instância do Excel reutilizada para todos os arquivos (evita abrir/fechar o processo repetidamente)
excel_app = win32.Dispatch("Excel.Application")
excel_app.Visible       = False
excel_app.DisplayAlerts = False

# Outlook só é iniciado em modo produção
outlook_app = None if MODO_TESTE else win32.Dispatch('outlook.application')

try:

    # ==========================
    # PROCESSO ESPECIALISTA
    # ==========================

    especialistas = df['ESPECIALISTA'].dropna().unique()
    print(f"Processando {len(especialistas)} especialistas...")

    for especialista in especialistas:

        df_esp = df[df['ESPECIALISTA'] == especialista].copy()
        email  = df_esp['EMAIL ESPECIALISTA'].iloc[0]

        nome_limpo   = limpar_nome(especialista)
        prefixo      = "TESTE_" if MODO_TESTE else ""
        nome_arquivo = os.path.join(PASTA_ATUAL, f"{prefixo}Especialista_{nome_limpo}_{data_nome}.xlsx")

        print(f"  [{tipo_label(MODO_TESTE)}] {especialista} → {len(df_esp)} registros")

        try:
            gerar_relatorio(df_esp, nome_arquivo, excel_app)

            if MODO_TESTE:
                registrar_log("Especialista", especialista, email, "GERADO")
                print(f"    ✅ Arquivo gerado: {nome_arquivo}")
            else:
                enviar_email(email, especialista, nome_arquivo, "Especialista", outlook_app)

        except Exception as e:
            registrar_log("Especialista", especialista, email, "ERRO", str(e))
            print(f"    ❌ ERRO: {e}")


    # ==========================
    # PROCESSO GERENTE
    # ==========================

    gerentes = df['GERENTE'].dropna().unique()
    print(f"\nProcessando {len(gerentes)} gerentes...")

    for gerente in gerentes:

        df_ger = df[df['GERENTE'] == gerente].copy()
        email  = df_ger['EMAIL GERENTE'].iloc[0]

        nome_limpo   = limpar_nome(gerente)
        prefixo      = "TESTE_" if MODO_TESTE else ""
        nome_arquivo = os.path.join(PASTA_ATUAL, f"{prefixo}Gerente_{nome_limpo}_{data_nome}.xlsx")

        print(f"  [{tipo_label(MODO_TESTE)}] {gerente} → {len(df_ger)} registros")

        try:
            gerar_relatorio(df_ger, nome_arquivo, excel_app)

            if MODO_TESTE:
                registrar_log("Gerente", gerente, email, "GERADO")
                print(f"    ✅ Arquivo gerado: {nome_arquivo}")
            else:
                enviar_email(email, gerente, nome_arquivo, "Gerente", outlook_app)

        except Exception as e:
            registrar_log("Gerente", gerente, email, "ERRO", str(e))
            print(f"    ❌ ERRO: {e}")

finally:
    # Garante que o Excel seja sempre encerrado, mesmo em caso de erro inesperado
    try:
        excel_app.Quit()
    except Exception:
        pass

    # Salva o log uma única vez ao final com todos os registros acumulados
    salvar_log()


if MODO_TESTE:
    print(f"\n✅ Modo teste concluído! Arquivos em: {PASTA_ATUAL}")
else:
    print("\n✅ Processo finalizado com sucesso!")